# MAVIS — Multimer-Aware Variant Impact Scoring
### Guided notebook for assessing missense-variant structural disruption in multimeric context

This notebook runs the MAVIS **variant-assessment** pipeline end-to-end from your browser, with a free GPU. For each missense variant it computes a **three-axis ΔΔG** profile — monomer fold, fold-in-complex, and binding interface — against AlphaFold structures, and reports a per-variant **mechanism call** and structural tier.

**What you do:** name a hub gene, its partner(s), and a list of variants. The notebook fetches or predicts the structures, builds the configuration, runs MAVIS, and shows readable per-variant **mechanism cards** (plus the full results table to download).

- **Primary output:** the structural assessment — *no external data required*.
- **Optional add-on:** four-way concordance, if you have manually-curated AlphaMissense + Franklin/ClinVar annotations.

> **Predict once, score many.** Predicted/fetched structures depend only on the gene pair, not on your variant list. Once the structures exist you can re-run the scoring step with a **new variant list without re-predicting** — just edit the variants and re-run Step 3.

Repo: https://github.com/la424/mavis · MAVIS reports *structural disruption and its mechanism*, not pathogenicity.

## ⚠️ Before you start: you must supply your own FoldX

**MAVIS uses [FoldX](https://foldxsuite.crg.eu/) to compute ΔΔG values, and FoldX is _not_ bundled with this notebook.** FoldX is **free for academic use** but requires its own registration/license.

**You must:**
1. Register and download FoldX (academic license) from **https://foldxsuite.crg.eu/**.
2. Get the **Linux** build (Colab runs Linux), and upload the `foldx` binary in the FoldX cell below.

Without your own FoldX binary the structural-scoring step cannot run. Everything before it (structure fetching/prediction, config building) still works.

In [ ]:
#@title Setup: clone/update MAVIS + install dependencies  { display-mode: "form" }
import os, sys, subprocess, shutil, importlib
from pathlib import Path

REPO = Path("/content/mavis")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/la424/mavis.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=False)  # pick up fixes on re-run

subprocess.run(["pip", "install", "-q", "pandas", "numpy", "biopython",
                "pyyaml", "openpyxl", "gemmi", "matplotlib"], check=True)

SCRIPTS = REPO / "scripts"
for d in (str(REPO / "notebooks"), str(SCRIPTS)):
    if d not in sys.path:
        sys.path.insert(0, d)
import mavis_helpers
importlib.reload(mavis_helpers)   # so re-runs use the freshly pulled code
H = mavis_helpers

WORK = Path("/content/work")
STRUCT = WORK / "structures"; STRUCT.mkdir(parents=True, exist_ok=True)
OUT = WORK / "out"
PY = sys.executable

gpu = subprocess.run(["bash", "-lc",
        "nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo none"],
        capture_output=True, text=True).stdout.strip()
print("Repo:", REPO)
print("GPU :", gpu if gpu and gpu != "none" else
      "none  (for ColabFold, set Runtime > Change runtime type > GPU)")
print("Setup OK.")

In [ ]:
#@title Upload your FoldX binary (LINUX build)  { display-mode: "form" }
from google.colab import files
print("Select your LINUX FoldX binary. Colab runs Linux, so a macOS or Windows FoldX build will NOT run here.")
up = files.upload()
fn = list(up.keys())[0]
FOLDX = WORK / "foldx"
shutil.move(fn, FOLDX)
os.chmod(FOLDX, 0o755)
os.environ["FOLDX_BINARY"] = str(FOLDX)

magic = open(FOLDX, "rb").read(4)
ELF = bytes([0x7f, 0x45, 0x4c, 0x46])
MACHO = (bytes([0xcf, 0xfa, 0xed, 0xfe]), bytes([0xce, 0xfa, 0xed, 0xfe]),
         bytes([0xca, 0xfe, 0xba, 0xbe]), bytes([0xfe, 0xed, 0xfa, 0xcf]))
if magic == ELF:
    try:
        subprocess.run([str(FOLDX), "-h"], capture_output=True, text=True, timeout=60)
        print("OK: Linux binary detected and runnable.")
        print("FOLDX_BINARY =", FOLDX)
    except OSError as e:
        print("Binary present but did not execute:", e)
elif magic in MACHO:
    print("ERROR: that is a macOS (Mach-O) FoldX binary - it cannot run on Colab's Linux.")
    print("Download the LINUX build from https://foldxsuite.crg.eu/ and upload that instead, then re-run this cell.")
elif magic[:2] == bytes([0x4d, 0x5a]):
    print("ERROR: that is a Windows (.exe) FoldX binary - it cannot run on Colab's Linux.")
    print("Download the LINUX build from https://foldxsuite.crg.eu/ and upload that instead, then re-run this cell.")
else:
    print("WARNING: this file does not look like a Linux executable (unexpected header).")
    print("Make sure you uploaded the LINUX FoldX binary from https://foldxsuite.crg.eu/.")

---
## Step 1 - Your hub gene, partners, and variants

**New here? Leave `USE_DEMO` checked** to run a built-in **TNNI3 + TNNC1** example (small, fast, known result) - just run every cell top to bottom. Uncheck `USE_DEMO` to analyze your own genes.

Fill in the form. List variants one per line in the `VARIANTS` list as `"GENE AAchange"` with 1-letter codes, e.g. `"SHROOM3 G1003R"`. You can mix hub and partner variants.

In [ ]:
#@title Variants & system  { display-mode: "form" }
USE_DEMO = True  #@param {type:"boolean"}
HUB_GENE = "SHROOM3"  #@param {type:"string"}
PARTNERS = "ROCK2"    #@param {type:"string"}
MONOMER_SOURCE = "AlphaFold DB (auto-fetch)"  #@param ["AlphaFold DB (auto-fetch)", "Manual upload"]
ORGANISM_ID = 9606    #@param {type:"integer"}

# One variant per line: "GENE AAchange"  (e.g. "SHROOM3 G1003R")
VARIANTS = [
    "SHROOM3 G1003R",
    "SHROOM3 T1012N",
]

import pandas as pd
from IPython.display import display, HTML
if USE_DEMO:
    HUB_GENE, PARTNERS, MONOMER_SOURCE = "TNNI3", "TNNC1", "AlphaFold DB (auto-fetch)"
    VARIANTS = ["TNNI3 R162W", "TNNI3 P82S"]
    print("DEMO MODE: TNNI3 + TNNC1 (small complex ~370 aa, known result). Uncheck USE_DEMO for your own inputs.")

partner_list = [p.strip() for p in PARTNERS.replace(",", " ").split() if p.strip()]
genes_all = [HUB_GENE] + partner_list
vdf, verr = H.parse_variants(chr(10).join(VARIANTS))
print("Hub:", HUB_GENE, "| Partners:", partner_list, "| Monomer source:", MONOMER_SOURCE)
print(f"Parsed {len(vdf)} variants across {vdf['gene'].nunique()} gene(s).")
display(vdf)
if verr:
    print("Skipped lines:")
    for ln, raw, why in verr:
        print("  line", ln, repr(raw), "->", why)
unknown = sorted(set(vdf["gene"]) - {g.lower() for g in genes_all})
if unknown:
    print("WARNING: these variant genes are not in your hub/partner list:", unknown)

## Step 2a — Monomer structures

Either **auto-fetch** each gene's model from the AlphaFold database (by UniProt), or **upload** your own monomer files (name each file to start with the gene symbol, e.g. `SHROOM3.pdb`).

In [ ]:
#@title Get monomer structures  { display-mode: "form" }
import shutil
gene_uniprot, gene_seq, gene_monomer = {}, {}, {}
try:
    ORGANISM_ID = int(str(ORGANISM_ID).strip())
except (ValueError, TypeError):
    ORGANISM_ID = 9606
    print("ORGANISM_ID was blank/invalid; defaulting to 9606 (human).")

if MONOMER_SOURCE == "AlphaFold DB (auto-fetch)":
    for g in genes_all:
        try:
            acc, cands = H.resolve_uniprot(g, organism_id=ORGANISM_ID)
        except Exception as e:
            print("Could not resolve", g, "on UniProt:", e, "- try Manual upload for this gene.")
            continue
        if not acc:
            print("WARNING: no reviewed UniProt entry for", g, "- use Manual upload for this gene.")
            continue
        try:
            path = H.fetch_alphafold_monomer(acc, STRUCT)
            gene_uniprot[g.lower()] = acc
            gene_monomer[g.lower()] = path
            gene_seq[g.lower()] = H.fetch_uniprot_sequence(acc)
            print(f"{g:10s} {acc}  {H.count_residues(path)} residues  -> {path.name}")
        except Exception as e:
            print("Resolved", g, "to", acc, "but could not fetch its AlphaFold model:", e)
else:
    from google.colab import files
    print("Upload one monomer file per gene (.pdb or .cif), each named to start with the gene symbol.")
    up = files.upload()
    for src in list(up.keys()):
        dest = STRUCT / src
        shutil.move(src, dest)
        stem = dest.stem.lower()
        match = next((g.lower() for g in genes_all if stem.startswith(g.lower())), None)
        if match:
            gene_monomer[match] = dest
            print(f"{match:10s} <- {dest.name}  ({H.count_residues(dest)} residues)")
        else:
            print("Could not map", dest.name, "to a gene; rename to start with the gene symbol.")
    for g in genes_all:
        if g.lower() not in gene_seq:
            try:
                acc, _ = H.resolve_uniprot(g, organism_id=ORGANISM_ID)
                if acc:
                    gene_uniprot[g.lower()] = acc
                    gene_seq[g.lower()] = H.fetch_uniprot_sequence(acc)
            except Exception:
                pass

print("Monomers ready for:", sorted(gene_monomer))

## Step 2b — The complex (size-routed)

MAVIS scores the binding interface in the assembled complex, so we need a structure of the hub with each partner. The notebook picks a route by size:

- **small / medium complex** -> predict with **ColabFold** on the GPU (one click);
- **large complex** (e.g. full-length SHROOM3 + a partner) -> use **AlphaFold Server** (predict at alphafoldserver.com, upload the `.cif` back here);
- **Domain-scoped ColabFold** -> model just the hub domain containing your variants (set `HUB_DOMAIN_RANGE` like `25-110`); MAVIS offsets are set automatically so full-length variant numbering still maps.

In [ ]:
#@title Predict / provide the complex structure  { display-mode: "form" }
MULTIMER_MODE = "Auto (recommended)"  #@param ["Auto (recommended)", "ColabFold (GPU)", "AlphaFold Server (upload .cif)", "Domain-scoped ColabFold"]
HUB_DOMAIN_RANGE = ""  #@param {type:"string"}

chain_ids = [chr(ord("A") + i) for i in range(len(genes_all))]
gene_chain = {g.lower(): chain_ids[i] for i, g in enumerate(genes_all)}
hub = HUB_GENE.lower()
mono_off = {g.lower(): 0 for g in genes_all}
multi_off = {g.lower(): 0 for g in genes_all}

seqs = []
for g in genes_all:
    s = gene_seq.get(g.lower(), "")
    if not s:
        print("WARNING: no sequence for", g, "- ColabFold/AF-Server need it.")
    if g.lower() == hub and HUB_DOMAIN_RANGE.strip():
        a, b = [int(x) for x in HUB_DOMAIN_RANGE.replace("-", " ").split()]
        s = H.slice_sequence(s, a, b)
        multi_off[hub] = a - 1     # complex is domain-scoped; full-length monomer stays offset 0
        print(f"Domain-scoping {HUB_GENE} to {a}-{b} ({len(s)} aa); multimer offset = {a-1}.")
    seqs.append(s)

total = sum(len(s) for s in seqs)
auto_route = H.route_by_size(total)
print(f"Total complex size ~{total} residues. Auto route: {auto_route}.")

mode = MULTIMER_MODE
if mode == "Auto (recommended)":
    mode = "ColabFold (GPU)" if auto_route == "colabfold" else "AlphaFold Server (upload .cif)"
    print("Selected:", mode)

complex_pdb = None
sys_name = "_".join(g.lower() for g in genes_all)

if mode in ("ColabFold (GPU)", "Domain-scoped ColabFold"):
    import importlib.util, glob as _glob
    if importlib.util.find_spec("colabfold") is None:
        print("Installing ColabFold (several minutes)...")
        subprocess.run(["bash", "-lc",
            "pip -q install 'colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold'"], check=True)
    fasta = H.write_fasta(sys_name, seqs, WORK / "cf_in")
    cf_out = WORK / "cf_out"; cf_out.mkdir(exist_ok=True)
    print("Running ColabFold on", fasta.name, "(can take a while on the free GPU)...")
    subprocess.run(["bash", "-lc", f"colabfold_batch --num-models 1 {fasta} {cf_out}"], check=True)
    hits = (_glob.glob(str(cf_out / "*relaxed_rank_001*.pdb"))
            or _glob.glob(str(cf_out / "*rank_001*.pdb"))
            or _glob.glob(str(cf_out / "*.pdb")))
    if not hits:
        print("ERROR: no ColabFold PDB produced; check the log above.")
    else:
        complex_pdb = STRUCT / f"{sys_name}.pdb"
        shutil.copy(sorted(hits)[0], complex_pdb)
        print("Complex model:", complex_pdb.name)

elif mode == "AlphaFold Server (upload .cif)":
    print("=== AlphaFold Server route (for large complexes) ===")
    print("1. Open https://alphafoldserver.com  (free; Google account).")
    print("2. New job; add ONE protein entity per chain, in THIS order:")
    for g, s in zip(genes_all, seqs):
        print(f"     chain {gene_chain[g.lower()]} = {g}  ({len(s)} aa)")
    seqfile = WORK / f"{sys_name}_sequences.txt"
    seqfile.write_text(chr(10).join(f">{g}{chr(10)}{s}" for g, s in zip(genes_all, seqs)))
    print("   (sequences saved to", seqfile, "for easy copy/paste)")
    print("3. Run it, download the .cif (or .zip), and upload here:")
    from google.colab import files
    up = files.upload()
    src = list(up.keys())[0]
    raw = WORK / src; shutil.move(src, raw)
    if raw.suffix.lower() == ".zip":
        import zipfile
        with zipfile.ZipFile(raw) as z:
            cif = [n for n in z.namelist() if n.lower().endswith(".cif")][0]
            z.extract(cif, WORK); raw = WORK / cif
    complex_pdb = STRUCT / f"{sys_name}.pdb"
    H.cif_to_pdb(raw, complex_pdb)
    print("Converted to PDB:", complex_pdb.name, "(", H.count_residues(complex_pdb), "residues )")

print("complex_pdb =", complex_pdb)

In [ ]:
#@title Build & validate the MAVIS config  { display-mode: "form" }
if complex_pdb is None:
    print("No complex structure yet - run the cell above first.")
else:
    genes_block = []
    for g in genes_all:
        gl = g.lower()
        if gl not in gene_monomer:
            print("WARNING: missing monomer for", g)
        genes_block.append({"gene": gl, "chain": gene_chain[gl],
                            "monomer_file": str(gene_monomer.get(gl, "")),
                            "monomer_offset": mono_off.get(gl, 0),
                            "multimer_offset": multi_off.get(gl, 0)})
    systems = [{"name": sys_name, "structure_type": "AF",
                "complex_file": str(complex_pdb), "genes": genes_block}]
    cfg_path = H.build_config_yaml(systems, WORK / "config.yaml")
    print(cfg_path.read_text())
    try:
        H.validate_config(cfg_path, SCRIPTS)
        print("Config valid (loaded by MAVIS).")
    except Exception as e:
        print("Config validation error:", e)

---
## Step 3 — Structural assessment (primary output)

Runs MAVIS with your FoldX binary, the fetched/predicted structures, and your variant list, then shows per-variant **mechanism cards** and a downloadable summary.

To score a **different variant list** later, just edit Step 1's `VARIANTS` and re-run this section — the structures are reused (no re-prediction).

In [ ]:
#@title Run MAVIS - structural assessment  { display-mode: "form" }
vcsv = WORK / "variants.csv"
vdf.to_csv(vcsv, index=False)
OUT.mkdir(parents=True, exist_ok=True)
env = dict(os.environ); env["FOLDX_BINARY"] = str(FOLDX)
cmd = [PY, str(REPO / "run.py"), "--config", str(WORK / "config.yaml"),
       "--variants", str(vcsv), "--structures", str(STRUCT),
       "--out", str(OUT), "--foldx", str(FOLDX)]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(proc.stdout[-3000:])
if proc.returncode != 0:
    print("STDERR (tail):", proc.stderr[-2000:])
res_csv = OUT / "structural_results.csv"
print("Results:", res_csv, "| exists:", res_csv.exists())

In [ ]:
#@title Results - mechanism cards, summary, downloads  { display-mode: "form" }
from IPython.display import display, HTML
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sdf = pd.read_csv(res_csv)
summary = H.per_partner_table(sdf)
display(HTML(H.render_cards_html(sdf)))
print("Per-variant x partner summary:")
display(summary)

gg = summary.copy()
gg["label"] = gg["gene"].str.upper() + " " + gg["variant"]
agg = gg.groupby("label").agg(
        monomer=("ddg_monomer", "first"),
        fold=("ddg_fold", lambda s: np.nanmax(np.abs(s)) if s.notna().any() else np.nan),
        binding=("ddg_binding", lambda s: np.nanmax(np.abs(s)) if s.notna().any() else np.nan))
ax = agg.plot(kind="barh", figsize=(7, 0.5 * len(agg) + 1))
ax.set_xlabel("max |ddG| (kcal/mol)")
ax.set_title("Per-variant structural effect by axis")
plt.tight_layout(); plt.savefig(WORK / "axes.png", dpi=130); plt.show()

sum_csv = OUT / "mavis_mechanism_summary.csv"
summary.to_csv(sum_csv, index=False)
from google.colab import files
print("Downloading full results + readable summary...")
files.download(str(res_csv))
files.download(str(sum_csv))

---
## Step 4 (optional) — Four-way concordance

Overlay your **manually-curated** AlphaMissense + Franklin/ClinVar calls next to the structural result. This is the only step that needs hand-entered data; skip it if you only want the structural assessment.

In [ ]:
#@title (Optional) Build the concordance template  { display-mode: "form" }
ctpl = H.concordance_template(vdf)
ctpl_path = OUT / "concordance_template.csv"
ctpl.to_csv(ctpl_path, index=False)
print("Fill the three columns (AM pathogenicity, AM class, franklin) per variant.")
print("Option A: edit ANNOTATIONS inline in the next cell. Option B: edit this CSV and upload it.")
display(ctpl)
from google.colab import files as _f
_f.download(str(ctpl_path))

In [ ]:
#@title (Optional) Run four-way concordance  { display-mode: "form" }
ANNOTATION_MODE = "Upload filled template"  #@param ["Upload filled template", "Use ANNOTATIONS dict below"]

ANNOTATIONS = [
    # {"gene":"shroom3","variant":"G1003R","AM pathogenicity":0.81,"AM class":"likely_pathogenic","franklin":"Likely Pathogenic"},
]

if ANNOTATION_MODE == "Upload filled template":
    from google.colab import files
    print("Upload filled .csv/.xlsx with columns: gene, variant, AM pathogenicity, AM class, franklin")
    up = files.upload()
    src = list(up.keys())[0]
    ann = pd.read_excel(src) if src.lower().endswith((".xlsx", ".xls")) else pd.read_csv(src)
else:
    ann = pd.DataFrame(ANNOTATIONS)

if len(ann) == 0:
    print("No annotations provided - skipping concordance.")
else:
    master, cproc = H.run_concordance(res_csv, ann, OUT / "concordance", SCRIPTS, python_exe=PY)
    print(cproc.stdout[-1500:])
    if master and master.exists():
        cdf = pd.read_csv(master)
        keep = [c for c in ["gene", "variant", "mavis_tier", "mavis_mechanism",
                            "AM pathogenicity", "AM class", "franklin"] if c in cdf.columns]
        display(cdf[keep])
        from google.colab import files as _f2
        _f2.download(str(master))
    else:
        print("Concordance produced no output:", cproc.stderr[-1000:])

---
## Notes, limits & troubleshooting

- **FoldX is required** and not bundled - upload your own Linux academic build (see the notice at the top).
- **GPU runtime for ColabFold:** Runtime > Change runtime type > GPU. The first ColabFold run also downloads model parameters and can take several minutes.
- **Large complexes** (e.g. full-length SHROOM3 + partner, ~3,000+ residues) exceed the free Colab GPU. Use the **AlphaFold Server** route (predict at alphafoldserver.com, upload the .cif - it is converted to PDB automatically), or **Domain-scoped ColabFold** to model just the domain containing your variants (offsets are set automatically so full-length numbering still maps).
- **Predict once, score many:** structures depend only on the gene pair. To score a new variant list, edit Step 1 and re-run Step 3 only.
- **Mechanism vs pathogenicity:** MAVIS reports structural disruption and its mechanism, not pathogenicity; the optional concordance overlays your curated AlphaMissense + Franklin/ClinVar calls.
- Built from https://github.com/la424/mavis